# CareReturn AI — Registry Measurement Noise

Simulate a measurement perturbation using training-derived feature scales.

800 synthetic patients, six encounters each. No real patient data or clinical validation.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from healthcare_ai.registry_loading import load_registry
from healthcare_ai.registry_labels import labeled_registry
from healthcare_ai.registry_split import registry_partitions
from healthcare_ai.settings import FEATURES,TARGET
from healthcare_ai.pipeline import build
from healthcare_ai.metrics import evaluate
raw=load_registry();frame=labeled_registry(raw)
train,val,test=registry_partitions(frame);model=build().fit(train[FEATURES],train[TARGET])


In [2]:
rng=np.random.default_rng(42);scale=train[FEATURES].std().to_numpy()
for level in [0,.2,.5]:
    x=val[FEATURES]+rng.normal(size=val[FEATURES].shape)*scale*level
    print(level,evaluate(val[TARGET],model.predict_proba(x)[:,1])['auroc'])

0 0.7055553204992595
0.2 0.7009858261053522
0.5 0.693750793314999


## Interpretation

All outcomes are simulator-generated. Repeated encounters from each patient remain in one partition. Clinical, prospective and external validation have not been performed. Thresholds and probability bands are educational examples.